# v3 DPO Training: v3 from v2.1-lite + preference pairs

Trains DPO on top of v2.1-lite using the preference pairs from v3_pair_gen.
Exports v3 as GGUF q4_k_m for local evaluation.

**Problem targeted:** Problem 2 (behavioural misses on in-distribution data).
**Runs on:** Kaggle GPU (T4 or P100).
**Input:** `data/qwen3.5_latest/dpo_pairs.jsonl` (from v3_pair_gen output)
**Output:** `v3_dpo-qwen3b-q4_k_m.gguf`

Setup instructions before running:
1. Add v3_pair_gen output as input (contains dpo_pairs.jsonl).
2. Add v2-1-lite-multi-template-sft-unsloth output as input (contains merged_fp16_v2_1_lite).

## Cell 1: env check

In [ ]:
import sys
print('Python:', sys.version)
!nvidia-smi

## Cell 2: clone repo

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

PAT  = UserSecretsClient().get_secret('GITHUB_PAT')
REPO = 'nizamphoenix/clinical_transcript_summariser'
DEST = '/kaggle/working/repo'

if os.path.exists(DEST):
    subprocess.run(['rm', '-rf', DEST], check=True)

url = f'https://{PAT}@github.com/{REPO}.git'
subprocess.run(['git', 'clone', '--depth', '1', url, DEST], check=True)
print('cloned ->', DEST)
%cd $DEST

## Cell 3: install dependencies

In [ ]:
!pip install -q unsloth
!pip uninstall unsloth -y
!pip install -q --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install -q -e .

import sys
sys.path.insert(0, '/kaggle/working/repo')
print('install done')

## Cell 4: load preference pairs

In [ ]:
import json
from pathlib import Path
from datasets import Dataset

# dpo_pairs.jsonl is saved to the repo data dir by v3_pair_gen,
# but on Kaggle it may arrive via a separate input dataset.
# Try repo data dir first, then /kaggle/input.
PAIRS_CANDIDATES = [
    Path('data/qwen3.5_latest/dpo_pairs.jsonl'),
    Path('/kaggle/input/v3-pair-gen/dpo_pairs.jsonl'),  # adjust name if needed
]

pairs_path = next((p for p in PAIRS_CANDIDATES if p.exists()), None)
assert pairs_path is not None, (
    "dpo_pairs.jsonl not found. Add v3_pair_gen output as a Kaggle input dataset."
)
print('loading pairs from', pairs_path)

with open(pairs_path) as f:
    pairs = [json.loads(l) for l in f if l.strip()]

# DPOTrainer needs prompt, chosen, rejected columns
ds = Dataset.from_list([
    {"prompt": p["prompt"], "chosen": p["chosen"], "rejected": p["rejected"]}
    for p in pairs
])

print(f'pairs loaded: {len(ds)}')

# Summary
from collections import Counter
modes = Counter(p['rejected_mode'] for p in pairs)
tmpls = Counter(p['template'] for p in pairs)
print('by template:', dict(tmpls))
print('by rejected_mode:', dict(modes))

## Cell 5: load v2.1 base + fresh LoRA

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
PatchDPOTrainer()

MODEL_PATH = "/kaggle/input/notebooks/nizamuddin/v2-1-lite-multi-template-sft-unsloth/merged_fp16_v2_1_lite"
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
print('base loaded')

# Attach a fresh LoRA (separate from the SFT adapter — clean separation)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=0,
)
model.print_trainable_parameters()

## Cell 5b: Sequence-length audit

Check what fraction of pairs would be truncated at `max_seq_length=1024`.
This matches the SFT baseline for fair comparison. Any pair exceeding 900 tokens
(90% of limit) is flagged — review the count before training.

In [ ]:
# Tokenise prompt+chosen and prompt+rejected to measure sequence lengths.
# DPOTrainer packs both into one forward pass; the longer of the two determines
# whether a pair is truncated.
# We check against max_seq_length=1024 (matching SFT baseline).
from pathlib import Path
import json
MAX_SEQ_LENGTH = 1024
WARN_THRESHOLD = 0.9  # flag pairs using >90% of budget
def load_jsonl(path: Path):
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)
## after generating pairs, they are uploaded to git, which is then dowloaded
pairs = list(load_jsonl(Path("/kaggle/working/repo/data/qwen3.5_latest/dpo_pairs.jsonl"))) 
print("pairs loaded:", len(pairs))
lengths = []
for p in pairs:
    combined_chosen   = p['prompt'] + p['chosen']
    combined_rejected = p['prompt'] + p['rejected']
    # Tokenise without adding special tokens to get raw length
    len_c = len(tokenizer.encode(combined_chosen,   add_special_tokens=False))
    len_r = len(tokenizer.encode(combined_rejected, add_special_tokens=False))
    lengths.append(max(len_c, len_r))

over_limit  = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)
near_limit  = sum(1 for l in lengths if l > MAX_SEQ_LENGTH * WARN_THRESHOLD)
mean_len    = sum(lengths) / len(lengths)
max_len     = max(lengths)

print(f'pairs checked       : {len(lengths)}')
print(f'mean max-side length: {mean_len:.0f} tokens')
print(f'max  max-side length: {max_len} tokens')
print(f'pairs > {MAX_SEQ_LENGTH} (truncated): {over_limit}  ({100*over_limit/len(lengths):.1f}%)')
print(f'pairs > {int(MAX_SEQ_LENGTH*WARN_THRESHOLD)} (near limit): {near_limit}  ({100*near_limit/len(lengths):.1f}%)')

if over_limit > 0:
    print(f'\nWARNING: {over_limit} pairs will be silently truncated at {MAX_SEQ_LENGTH}.')
    print('These are consistent with the SFT training budget; no config change needed.')
    print('If >20% of pairs are truncated, consider raising max_seq_length and checking T4 OOM.')
else:
    print('\nAll pairs fit within max_seq_length=1024. No truncation risk.')


## Cell 6: DPO training

In [ ]:
from datasets import Dataset
ds = Dataset.from_list(
    [{"prompt": p["prompt"], "chosen": p["chosen"], "rejected": p["rejected"]} for p in pairs]
)
# create train and val split
ds = ds.shuffle(seed=0)
split = ds.train_test_split(test_size=0.05, seed=0)  # ~10 eval pairs
## The signal we want is "is the model collapsing or diverging globally?" for this small dataset. we would stratify this in production case by template
train_ds = split["train"]
eval_ds  = split["test"]

In [ ]:
from trl import DPOTrainer, DPOConfig

# Conservative hyperparameters: low beta keeps KL anchor tight,
# preventing the model from trading misses for hallucinations.
# Tuned to run on a single T4 on Kaggle.

# Step budget (full ~213-pair set, 95/5 split -> ~202 train):
#   steps_per_epoch = ceil(len(train_ds) / (per_device_train_batch_size * gradient_accumulation_steps))
#                   = ceil(202 / (2 * 4)) = 26
#   total_steps     = num_train_epochs * steps_per_epoch = 4 * 26 = ~104
#   warmup_steps    = 0.1 * total_steps = ~10
EVAL_EVERY = 10
config = DPOConfig(
    beta=0.1,
    learning_rate=5e-6,
    num_train_epochs=4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    eval_strategy="steps",
    eval_steps=EVAL_EVERY,
    per_device_eval_batch_size=2,
    lr_scheduler_type='cosine',
    warmup_steps=10,  # ~10% of ~104 total steps (see calc above)
    seed=0,
    max_length=1024,
    max_prompt_length=700,
    output_dir='dpo_out_v3',
    logging_steps=EVAL_EVERY,
    save_strategy='steps',
    save_steps=EVAL_EVERY,
    save_total_limit=1,         # keep disk footprint minimal (adapters only)
    load_best_model_at_end=True,
    report_to='none',
)

trainer = DPOTrainer(
    model=model,
    args=config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
)

print('starting DPO training...')
trainer.train()
print('training complete')

## Cell 7: sanity check before export

Run the trained model on a few in-distribution transcripts and verify
`schema_valid=1` on all outputs. If any fail, investigate before exporting.

In [ ]:
from src.verifier import score_prediction
from src.prompts import build_inference_messages
from src.data_generation.templates import REGISTRY
import json

FastLanguageModel.for_inference(model)

DATA_ROOT = '/kaggle/working/repo/data/qwen3.5_latest' 

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]


sanity_rows = []
for tpl, fname in [('soap', 'eval_in_dist.soap.jsonl'),
                   ('referral_a', 'eval_in_dist.referral_a.jsonl'),
                   ('mse', 'eval_in_dist.mse.jsonl')]:
    rows = load_jsonl(f'{DATA_ROOT}/{fname}')
    for r in rows[:2]: # Pick 2 samples per trained template for sanity check
        sanity_rows.append(dict(r, template=tpl))

print(f'sanity checking on {len(sanity_rows)} samples...')
print()
all_ok = True
for row in sanity_rows:
    template   = row['template']
    label_key  = REGISTRY[template]['label_key']
    gold       = row[label_key]
    transcript = row['transcript']

    messages = build_inference_messages(template, transcript)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs, 
        max_new_tokens=1024, 
        do_sample=False,  # greedy
        #temperature=0.1, #cannot be 0.0
        pad_token_id=tokenizer.eos_token_id
    )
    raw = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    sc = score_prediction(template, raw, gold, transcript)
    status = 'OK' if sc['schema_valid'] else 'FAIL'
    if not sc['schema_valid']:
        all_ok = False
    print(f'  [{status}] template={template}  schema_valid={sc["schema_valid"]}  '
          f'overlap={sc["content_overlap"]:.2f}  wrong_null={sc["wrong_null"]}/{sc["gold_filled"]}')

print()
if all_ok:
    print('All sanity checks passed. Safe to export.')
else:
    print('WARNING: schema_valid=0 on some outputs. Investigate before exporting.')

## Cell 8: merge LoRA and export GGUF q4_k_m

In [ ]:
# # Merge LoRA into the base weights
# model.save_pretrained_merged('merged_fp16_v3', tokenizer, save_method='merged_16bit')
# print('merged weights saved -> merged_fp16_v3/')

# # Export to GGUF q4_k_m
# # Unsloth names the file: v3_dpo-unsloth.Q4_K_M.gguf
# model.save_pretrained_gguf('v3_dpo', tokenizer, quantization_method='q4_k_m')
# print('GGUF saved')
# print()
# print('Download the GGUF and rename to: v3_dpo-qwen3b-q4_k_m.gguf')
# print('Place in: models/ in the local repo')
# !ls -lh *.gguf 2>/dev/null || echo 'no gguf in cwd, check Kaggle output panel'

In [ ]:
# ---------------------------------------------------------------------------
#  (1) drop the redundant save_pretrained_merged call,
#  (2) clear training checkpoints first,
#  (3) delete intermediates after, keeping only the q4_k_m file.
# ---------------------------------------------------------------------------
import shutil, glob, os

# (1) Training checkpoints are dead weight now: load_best_model_at_end=True has
#     already restored the best weights into memory.
shutil.rmtree('dpo_out_v3', ignore_errors=True)
print('cleared dpo_out_v3/ checkpoints')
print('--- disk before export ---')
!df -h /kaggle/working | tail -1

# (2) Export straight to GGUF q4_k_m. Do NOT also call save_pretrained_merged():
#     save_pretrained_gguf performs its own internal 16-bit merge. Calling both
#     is what exhausted the disk last time.
model.save_pretrained_gguf('v3_dpo', tokenizer, quantization_method='q4_k_m')
print('GGUF saved')
print('--- disk after gguf ---')
!df -h /kaggle/working | tail -1

# (3) Keep only the q4_k_m file. Remove the fp16 HF merge dir and the F16 gguf,
#     but never delete a directory that actually contains the q4 file.
q4_files = glob.glob('**/*[qQ]4_[kK]_[mM]*.gguf', recursive=True)
print('q4_k_m file(s):', q4_files)
assert q4_files, 'No q4_k_m gguf found - aborting cleanup so nothing is lost.'

protected = {os.path.abspath(f) for f in q4_files}

# Remove the fp16 HF safetensors merge dir only if it holds no q4 file.
if not any(p.startswith(os.path.abspath('v3_dpo') + os.sep) for p in protected):
    shutil.rmtree('v3_dpo', ignore_errors=True)
    print('removed fp16 HF merge dir: v3_dpo/')

# Remove F16 gguf intermediates (these are never the q4 file).
for f in glob.glob('**/*[fF]16*.gguf', recursive=True):
    if os.path.abspath(f) not in protected:
        os.remove(f)
        print('removed F16 intermediate:', f)

print('--- disk after cleanup ---')
!df -h /kaggle/working | tail -1
print()
print('Download the q4_k_m gguf and rename to: v3_dpo-qwen3b-q4_k_m.gguf')
print('Place in: models/ in the local repo')
!ls -lh **/*.gguf 2>/dev/null || echo 'no gguf found, check Kaggle output panel'